# 🎙️ Wu OmniVoice Studio — Google Colab GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wukongdegen/Wu-OmniVoice-Colab/blob/main/Wu_OmniVoice_Colab.ipynb)

Hệ thống tạo giọng nói AI siêu thực, Clone giọng mẫu, Thiết kế giọng theo thuộc tính và Tự động ghép Audio/Video, tạo Motion Video từ ảnh bằng GPU.

> ⚠️ **LƯU Ý QUAN TRỌNG:** Hãy chắc chắn đã bật GPU trước khi chạy:
> **Runtime** -> **Change runtime type** -> Chọn **T4 GPU** (hoặc **L4 / A100**) -> Bấm **Save**.

In [ ]:
#@title 🚀 KHỞI CHẠY WU OMNIVOICE STUDIO 【BẤM NÚT PLAY ĐỂ BẮT ĐẦU】
#@markdown ---
#@markdown ### ⚙️ **Cấu hình hệ thống:**
Luu_Vao_Google_Drive = True #@param {type:"boolean"}
Bat_Cloudflare_Tunnel = False #@param {type:"boolean"}
Bat_Gradio_Share = True #@param {type:"boolean"}
Port = 7860 #@param {type:"integer"}
#@markdown ---

RELEASE_BASE_URL = "https://github.com/wukongdegen/Wu-OmniVoice-Colab/releases/download/v0.2.4"

import os
import sys
import subprocess
import shutil
from pathlib import Path

PY_TAG = f"cp{sys.version_info.major}{sys.version_info.minor}"
if PY_TAG not in {"cp312", "cp313"}:
    print(f"❌ Python {sys.version_info.major}.{sys.version_info.minor} chưa được hỗ trợ. Hãy dùng runtime Colab Python 3.12 hoặc 3.13.")
    sys.exit(1)
WHEEL_NAMES = {
    "cp312": "wu_omnivoice-0.1.5-cp312-cp312-linux_x86_64.whl",
    "cp313": "wu_omnivoice-0.2.4-cp313-cp313-linux_x86_64.whl",
}
WHEEL_NAME = WHEEL_NAMES[PY_TAG]
WHEEL_URL = f"{RELEASE_BASE_URL}/{WHEEL_NAME}"

print("=" * 65)
print("  🌟 WU OMNIVOICE STUDIO - KHỞI TẠO HỆ THỐNG CLOUD GPU")
print("=" * 65)

def run_cmd(cmd_str, desc):
    res = subprocess.run(cmd_str, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
    if res.returncode != 0:
        print(f"   ⚠️ Cảnh báo ({desc}): {res.stderr.decode('utf-8', errors='ignore')[:200]}")
    return res.returncode

# 1. Kiểm tra GPU CUDA
print("\n[1/5] 🔍 Đang kiểm tra GPU...")
try:
    import torch
    if not torch.cuda.is_available():
        print("❌ LỖI: Chưa bật GPU! Vui lòng chọn Runtime -> Change runtime type -> T4 GPU rồi chạy lại.")
        sys.exit(1)
    print(f"   ✅ Đã nhận diện GPU: {torch.cuda.get_device_name(0)} (CUDA Sẵn sàng)")
except Exception as e:
    print(f"   ⚠️ Kiểm tra GPU: {e}")

# 2. Cấu hình thư mục lưu trữ
print("\n[2/5] 💾 Thiết lập thư mục lưu trữ dữ liệu...")
if Luu_Vao_Google_Drive:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        base_dir = Path("/content/drive/MyDrive/WuOmniVoice")
    except Exception as e:
        print(f"   ⚠️ Không thể mount Drive ({e}), chuyển sang lưu tạm /content/WuOmniVoice")
        base_dir = Path("/content/WuOmniVoice")
else:
    base_dir = Path("/content/WuOmniVoice")
base_dir.mkdir(parents=True, exist_ok=True)
(base_dir / "voice_store" / "audio").mkdir(parents=True, exist_ok=True)
(base_dir / "Audio Output").mkdir(parents=True, exist_ok=True)
os.environ["OMNIVOICE_BASE_DIR"] = str(base_dir)
cache_dir = base_dir / "cache"
cache_dir.mkdir(parents=True, exist_ok=True)
os.environ["PIP_CACHE_DIR"] = str(cache_dir / "pip")
os.environ["HF_HOME"] = str(cache_dir / "huggingface")
os.environ["HUGGINGFACE_HUB_CACHE"] = str(cache_dir / "huggingface" / "hub")
print(f"   ✅ Thư mục dữ liệu: {base_dir}")

# 3. Cài đặt hệ thống (ffmpeg). Các Python deps sẽ do wheel tự kéo về (install_requires).
print("\n[3/5] 📦 Cài đặt hệ thống (FFmpeg)...")
if shutil.which("ffmpeg") and shutil.which("ffprobe"):
    print("   ✅ FFmpeg đã có sẵn; bỏ qua cài đặt.")
else:
    rc = run_cmd("apt-get update -qq && apt-get install -y -qq ffmpeg", "Cài đặt FFmpeg")
    if rc != 0:
        print("   ❌ LỖI: Không thể cài FFmpeg.")
        sys.exit(1)
print("   ✅ FFmpeg sẵn sàng.")

# 4. Tải & cài đặt gói Binary Wu OmniVoice (.whl) — CHỈ chạy từ binary, KHÔNG có fallback mã nguồn.
print("\n[4/5] ⚡ Tải & cài đặt gói phần mềm Wu OmniVoice (Binary)...")
whl_path = str(cache_dir / WHEEL_NAME)
if not os.path.exists(whl_path):
    print("   ⬇️ Đang tải gói binary...")
    rc = run_cmd(f'curl -fSL -o {whl_path} "{WHEEL_URL}"', "Tải wheel")
    if rc != 0 or not os.path.exists(whl_path) or os.path.getsize(whl_path) < 10000:
        print("   ❌ LỖI: Không tải được gói phần mềm. Vui lòng liên hệ nhà phát hành để lấy link mới.")
        sys.exit(1)
installed_version = subprocess.run(
    [sys.executable, "-c", "from importlib.metadata import version; print(version('wu_omnivoice'))"],
    capture_output=True, text=True,
).stdout.strip()
required_version = WHEEL_NAME.split("-")[1]
check = subprocess.run([sys.executable, "-c", "import omnivoice, omnivoice.cli.local_web; print('ok')"], capture_output=True, text=True)
if "ok" not in check.stdout or installed_version != required_version:
    print(f"   📦 Cài/cập nhật OmniVoice {required_version} từ wheel cache...")
    rc = run_cmd(f'{sys.executable} -m pip install -q --upgrade "{whl_path}"', "Cài đặt Wheel")
    if rc != 0:
        print("   ❌ LỖI: Không thể cài đặt gói phần mềm.")
        sys.exit(1)
else:
    print(f"   ✅ OmniVoice {installed_version} đã có sẵn; bỏ qua cài đặt lại.")

# Xác minh cài đặt thành công (import được package binary)
check = subprocess.run(
    [sys.executable, "-c", "import omnivoice, omnivoice.cli.local_web; print('ok')"],
    capture_output=True, text=True,
)
if "ok" not in check.stdout:
    print("   ❌ LỖI: Gói binary không tương thích runtime này.")
    print(f"      Python runtime: {sys.version}")
    print(f"      Chi tiết đầy đủ:\n{check.stderr}")
    print("      Hãy kiểm tra dependency bị thiếu ở dòng lỗi cuối cùng.")
    sys.exit(1)
print("   ✅ Đã nạp gói phát hành Binary (.whl)")

# 5. Cloudflare Tunnel (tuỳ chọn)
if Bat_Cloudflare_Tunnel and not os.path.exists("/tmp/cloudflared"):
    run_cmd("curl -sLo /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /tmp/cloudflared", "Tải Cloudflare")

# 6. Khởi chạy Web UI Gradio
print("\n[5/5] 🚀 Đang khởi động Web UI...")
cmd_args = [
    sys.executable, "-c", "import sys, omnivoice.cli.local_web as lw; sys.exit(lw.main(sys.argv[1:]))",
    "--ip", "0.0.0.0",
    "--port", str(Port),
    "--device", "cuda:0",
]
if Bat_Gradio_Share:
    cmd_args.append("--share")
if Bat_Cloudflare_Tunnel:
    cmd_args.append("--tunnel")

print("=" * 65)
print("⚠️ Web UI đang khởi động model; chỉ hiển thị link sau khi Gradio bind thành công.")
print("=" * 65 + "\n")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
process = subprocess.Popen(
    cmd_args,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="", flush=True)
res = process.wait()
if res != 0:
    raise RuntimeError(f"Web UI failed to start (exit code {res}). The full startup log is shown above.")


### 📁 Công cụ phụ trợ: Tải về toàn bộ Audio & Video đã tạo
*(Dùng trong trường hợp bạn không bật liên kết Google Drive và muốn tải file về máy tính)*

In [ ]:
#@title 📦 Nén và Tải về file kết quả (.ZIP) 【Bấm Play】
import os
import shutil
from google.colab import files

base_dir = os.environ.get("OMNIVOICE_BASE_DIR", "/content/WuOmniVoice")
output_dir = os.path.join(base_dir, "Audio Output")
zip_filename = "/content/Wu_OmniVoice_Output.zip"

if os.path.exists(output_dir) and len(os.listdir(output_dir)) > 0:
    print(f"📦 Đang nén các file trong {output_dir}...")
    shutil.make_archive("/content/Wu_OmniVoice_Output", 'zip', output_dir)
    print("⬇️ Đang tải file zip về máy của bạn...")
    files.download(zip_filename)
else:
    print("⚠️ Chưa có file nào trong thư mục Audio Output để tải về.")
